# Brain Tumor MRI — train on Colab

Runtime → Change runtime type → **GPU** (T4 is enough). Running all cells trains the three backbones and builds the comparison table (about 30–45 minutes on a T4).

In [ ]:
!git clone https://github.com/krichenyoussef/data_cleaning-.git brain-tumor-mri
%cd brain-tumor-mri
!pip install -q -r requirements.txt

## 1. Download the dataset

Upload your `kaggle.json` (Kaggle → Settings → API → *Create New Token*) when prompted.

In [ ]:
from google.colab import files
files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p data/raw --unzip
!ls data/raw

## 2. Remove duplicates and build a leak-free split

In [ ]:
!python -m src.dedup
!cat results/dedup_report.json

## 3. Train and compare the three backbones

In [ ]:
!python -m src.train --model all --epochs 15 --batch-size 32

In [ ]:
!python -m src.compare

## 4. Grad-CAM examples

In [ ]:
for m in ['resnet50', 'efficientnet_b0', 'vit_small']:
    !python -m src.gradcam --checkpoint checkpoints/{m}.pt

from IPython.display import Image, display
display(Image('results/resnet50/gradcam_examples.png'))

## 5. Save the results

Download `results/` and commit it to the repo (the figures and `RESULTS.md` go in the README). Keep the best checkpoint for the demo (`checkpoints/best.pt`).

In [ ]:
!zip -qr results.zip results
files.download('results.zip')